#**GeoMine Sentinel** : Guarding land, law, and life from above.
---
GeoMine Sentinel: An AI-powered satellite monitoring system that detects mining footprints and cross-verifies them against legal and geospatial records to flag potential unauthorized mining.

'while the world looks up to clean energy as the uncompromisable future, througout the regions of devloping/underdevloped nations prevail the harvestation of energy at the cost of human life, environment and human future. The practice of unauthorized mining is still relevent in few regions far from ordinary livelyhood, we solve the problem by building a system that identify unauthorized mining footprint.

##importing necessary libraries

In [ ]:
!pip install torch transformers datasets evaluate accelerate geopandas rasterio earthengine-api geemap

In [ ]:
import torch
from huggingface_hub import login, whoami
from transformers import (
    SegformerImageProcessor, #AutoImageProcessor
    SegformerForImageClassification, #AutoModel
    TrainingArguments,
    Trainer,
    AutoTokenizer,
    pipeline
)
from huggingface_hub import login, whoami
#from evaluate import load
from datasets import load_dataset
from PIL import Image
import numpy as np, pandas as pd, matplotlib.pyplot as plt

##login()

In [ ]:
login()

In [ ]:
whoami()

In [ ]:
import torch
device=torch.device('cuda'if torch.cuda.is_available() else 'cpu')
print(device)

cuda


## Dataset: [Global mining footprint mapped from high-resolution satellite imagery by Zenodo](https://zenodo.org/records/6806817)
---
The dataset consists of mining polygons


1. Mining polygons: the existing dataset consists of the location of the mining footprints
2. Satellite Imagery: We must use the location to download the mining footprints from satellite imagery, preferably from Sentinel-2 or Google Earth
3. Combining: We then combine the polygons with the satellite images.
4. We feed the downloaded images to a segmentation model and compare its prediction to the polygons



In [3]:
import os
import geopandas as gpd

In [4]:
shp_path=r'/content/drive/MyDrive/Supplementary 1：mine area polygons/74548 mine polygons/74548_projected.shp'
mines=gpd.read_file(shp_path)

In [5]:
print(f'''
No. of polygons : {len(mines)}
Coordinate System : {mines.crs}
Columns : {mines.columns.tolist()}
{mines.head()}''')


No. of polygons : 74548
Coordinate System : PROJCS["WGS 84 / NSIDC EASE-Grid Global",GEOGCS["WGS 84",DATUM["WGS_1984",SPHEROID["WGS 84",6378137,298.257223563,AUTHORITY["EPSG","7030"]],AUTHORITY["EPSG","6326"]],PRIMEM["Greenwich",0],UNIT["Degree",0.0174532925199433]],PROJECTION["Cylindrical_Equal_Area"],PARAMETER["standard_parallel_1",30],PARAMETER["central_meridian",0],PARAMETER["false_easting",0],PARAMETER["false_northing",0],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH]]
Columns : ['OBJECTID', 'Name', 'Shape_Le_1', 'Shape_Area', 'geometry']
   OBJECTID Name    Shape_Le_1    Shape_Area  \
0         1   Au    507.731122  1.213696e+04   
1         2   Au  22248.346547  6.566996e+06   
2         3   Au  19467.203319  7.109258e+06   
3         4   Fe   8281.186849  2.323061e+06   
4         5    0   1492.049584  1.231474e+05   

                                            geometry  
0  POLYGON Z ((9746741.997 3480087.392 0, 9746759...  
1  POLYGON Z

In [6]:
import ee #Google Earth Engine will be used apply filter to Sentinal-2 Data
ee.Authenticate() #authenticating EarthEngine

In [ ]:
ee.Initialize(project='mining-footprint')
print(ee.String("Connected!").getInfo())

In [8]:
# LOADING A SENTINAL-2 IMAGE COMPLETELY
point = ee.Geometry.Point([87.1150,23.6289]) #Raniganj Caol field cooardinates #([longitude,latitude])
region=point.buffer(5000).bounds() #creates a region around the point
images=(
    ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
    .filterBounds(region)
    .filterDate("2026-01-01", "2026-10-13")
    .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 10))
)
print('Image Found : ',images.size().getInfo())


Image Found :  93


In [10]:
# DISPLAYONG A SATALITE IMAGE
image= ee.Image(images.sort('CLOUD_PIXEL_PERCENTAGE').first())
print('Image Date ',ee.Date(image.get('system:time_start')).format('dd-MM-YYYY').getInfo())
import geemap #display rgb satalite images in notebook
map=geemap.Map(basemap='SATELLITE')
map.centerObject(region,12)
map.addLayer(
    image,
    {
        'bands':['B4','B3','B2'],
        'min':0,
        'max':3000
    },
    'Sentinal-2 RGB'
)
#map.addLayer(region,{},'Study area')
map

Image Date  01-01-2026


Map(center=[23.628906035609788, 87.11507921814916], controls=(WidgetControl(options=['position', 'transparent_…

In [18]:
bands=image.bandNames().getInfo()
print('Available bands : ',bands)
print('len : ',len(bands))

for band in bands:
  map.addLayer(
      image.select(band),
      {
          'min':0,
          'max':3000,
      },
      band, #The bands own name become the label
      False,
  )
map

Available bands :  ['B1', 'B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B8', 'B8A', 'B9', 'B11', 'B12', 'AOT', 'WVP', 'SCL', 'TCI_R', 'TCI_G', 'TCI_B', 'MSK_CLDPRB', 'MSK_SNWPRB', 'QA10', 'QA20', 'QA60', 'MSK_CLASSI_OPAQUE', 'MSK_CLASSI_CIRRUS', 'MSK_CLASSI_SNOW_ICE']
len :  26


Map(bottom=28601.0, center=[23.82052617508277, 87.2064237211602], controls=(WidgetControl(options=['position',…

In [17]:
value=image.reduceRegion(
    reducer=ee.Reducer.first(),
    geometry=image.geometry().centroid(maxError=1),
    scale=20,
).getInfo()
print(value)

{'AOT': 381, 'B1': 655, 'B11': 2602, 'B12': 1643, 'B2': 775, 'B3': 964, 'B4': 1115, 'B5': 1410, 'B6': 1845, 'B7': 2054, 'B8': 2303, 'B8A': 2257, 'B9': 2195, 'MSK_CLASSI_CIRRUS': 0, 'MSK_CLASSI_OPAQUE': 0, 'MSK_CLASSI_SNOW_ICE': 0, 'MSK_CLDPRB': 0, 'MSK_SNWPRB': 0, 'QA10': None, 'QA20': None, 'QA60': 0, 'SCL': 5, 'TCI_B': 80, 'TCI_G': 98, 'TCI_R': 114, 'WVP': 1166}


###Downloading the mining footprints for India

In [22]:
print(mines.geometry)

0        POLYGON Z ((9746741.997 3480087.392 0, 9746759...
1        POLYGON Z ((10362412.948 5571579.856 0, 103631...
2        POLYGON Z ((-6976558.709 -1779192.275 0, -6976...
3        POLYGON Z ((11034240.022 5539946.449 0, 110342...
4        POLYGON Z ((8889342.751 3577269.676 0, 8889568...
                               ...                        
74543    POLYGON Z ((11805979.146 4676473.655 0, 118060...
74544    POLYGON Z ((11805247.073 4676355.488 0, 118052...
74545    POLYGON Z ((11806804.354 4677468.01 0, 1180675...
74546    POLYGON Z ((11806623.082 4677715.836 0, 118066...
74547    POLYGON Z ((11806252.132 4677565.694 0, 118061...
Name: geometry, Length: 74548, dtype: geometry


In [19]:
print(mines.crs) #coordinate system

PROJCS["WGS 84 / NSIDC EASE-Grid Global",GEOGCS["WGS 84",DATUM["WGS_1984",SPHEROID["WGS 84",6378137,298.257223563,AUTHORITY["EPSG","7030"]],AUTHORITY["EPSG","6326"]],PRIMEM["Greenwich",0],UNIT["Degree",0.0174532925199433]],PROJECTION["Cylindrical_Equal_Area"],PARAMETER["standard_parallel_1",30],PARAMETER["central_meridian",0],PARAMETER["false_easting",0],PARAMETER["false_northing",0],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH]]


In [21]:
mines.head()

,OBJECTID,Name,Shape_Le_1,Shape_Area,geometry
0,1,Au,507.731122,1.213696e+04,"POLYGON Z ((9746741.997 3480087.392 0, 9746759..."
1,2,Au,22248.346547,6.566996e+06,"POLYGON Z ((10362412.948 5571579.856 0, 103631..."
2,3,Au,19467.203319,7.109258e+06,"POLYGON Z ((-6976558.709 -1779192.275 0, -6976..."
3,4,Fe,8281.186849,2.323061e+06,"POLYGON Z ((11034240.022 5539946.449 0, 110342..."
4,5,0,1492.049584,1.231474e+05,"POLYGON Z ((8889342.751 3577269.676 0, 8889568..."


###Filtering Sentinal-2 Images

In [42]:
from shapely.geometry import mapping

# STEP 1: Convert ALL mine polygons to longitude/latitude.
# This creates a GeoDataFrame, preserving all rows and attributes.
mines_wgs84 = mines.to_crs("EPSG:4326")

# STEP 2: Get the bounding box of EACH polygon.
# The result is a DataFrame with minx, miny, maxx, and maxy columns.
bounds = mines_wgs84.geometry.bounds

# STEP 3: Keep polygons whose bounding boxes overlap India's approximate extent.
india_candidates = mines_wgs84[
    (bounds["maxx"] >= 68) &
    (bounds["minx"] <= 98) &
    (bounds["maxy"] >= 6) &
    (bounds["miny"] <= 38)
].copy()

print("Original polygons:", len(mines))
print("India-region candidates:", len(india_candidates))

# STEP 4: Select ONE candidate polygon to test.
# This is a row from the filtered GeoDataFrame, not the original dataset.
mine = india_candidates.iloc[0]

# STEP 5: Add a 150-metre buffer.
# IMPORTANT: first transform this polygon to a projected CRS measured in metres.
mine_projected = gpd.GeoSeries(
    [mine.geometry],
    crs="EPSG:4326"
).to_crs(mines.crs).iloc[0]

mine_geo = gpd.GeoSeries(
    [mine_projected],
    crs=mines.crs
).buffer(150).iloc[0]

# STEP 6: Convert the buffered polygon back to longitude/latitude.
mine_long_lat = gpd.GeoSeries(
    [mine_geo],
    crs=mines.crs
).to_crs("EPSG:4326").iloc[0]

# STEP 7: Convert the polygon to an Earth Engine geometry.
roi = ee.Geometry(mapping(mine_long_lat))

# STEP 8: Display information about the selected polygon.
print("Mine ID:", mine["OBJECTID"])
print("Mine type/name:", mine["Name"])
print("Bounds (longitude/latitude):", mine_long_lat.bounds)

Original polygons: 74548
India-region candidates: 1913
Mine ID: 5
Mine type/name: 0
Bounds (longitude/latitude): (92.12822649112681, 29.262744989214315, 92.13599099561145, 29.268950401886126)


In [43]:
s2=ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
s2=s2.filterBounds(roi)
s2=s2.filterDate('2026-04-01','2026-05-31')
s2=s2.filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE',30))
print('Matching Sentinal-2 images :',s2.size().getInfo())

Matching Sentinal-2 images : 1


###Selecting the least cloudy image

In [44]:
def mask_s2(image):
  scl=image.select('SCL') #Scene Classification Layer (SCL)
  clear=(
      scl.neq(3)
      .And(scl.neq(8))
      .And(scl.neq(9))
      .And(scl.neq(10))
      .And(scl.neq(11))
  )
  return image.updateMask(clear)

s2=s2.map(mask_s2)
s2=s2.sort('CLOUDY_PIXEL_PERCENTAGE')
if s2.size().getInfo()>0:
  best_img=ee.Image(s2.first())
  print('Date :',ee.Date(best_img.get('system:time_start')).format('dd-MM-YYYY').getInfo())
  print('Granular cloud cover :',best_img.get('CLOUDY_PIXEL_PERCENTAGE').getInfo(),'%')
else :
  print('No img found. Try a wider date range or a higher cloud limit.')

Date : 20-05-2026
Granular cloud cover : 1.243559 %


##Preprocessing & Augmentation

In [ ]:
processor = SegformerImageProcessor.from_pretrained('nvidia/mit-b0')
print(processor)
